# 2D/3D registration with iterative optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eigenvivek/nanodrr/blob/main/tutorials/optim.ipynb)

In [ ]:
!uv pip install nanodrr

In [ ]:
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

from nanodrr.camera import make_k_inv, make_rt_inv
from nanodrr.data import Subject, download_deepfluoro
from nanodrr.drr import render
from nanodrr.metrics import NormalizedCrossCorrelation2d
from nanodrr.plot import plot_drr
from nanodrr.registration import Registration

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Load the volume
imagepath, _ = download_deepfluoro(subject=1)
subject = Subject.from_filepath(imagepath).to(device)

# Construct the inverse intrinsic matrix from C-arm imaging parameters
sdd = 1020.0
delx = dely = 2.0
x0 = y0 = 0.0
height = width = 200

k_inv = make_k_inv(sdd, delx, dely, x0, y0, height, width, device=device)
sdd = torch.tensor([sdd], device=device)

In [ ]:
# Make the ground truth and initial poses
rt_inv_true = make_rt_inv(
    rotation=torch.tensor([[0.0, 0.0, 0.0]]),
    translation=torch.tensor([[0.0, 850.0, 0.0]]),
    orientation="AP",
    isocenter=subject.isocenter.cpu(),
).to(dtype=torch.float32, device=device)

rot = torch.tensor([[-0.1303,  0.3461, -0.7852]]) / torch.pi * 180
xyz = torch.tensor([[-11.8600, 828.8053, -24.4597]])
rt_inv_pred = make_rt_inv(
    rotation=rot,
    translation=xyz,
    orientation="AP",
    isocenter=subject.isocenter.cpu(),
).to(dtype=torch.float32, device=device)

# Plot
true = render(subject, k_inv, rt_inv_true, sdd, height, width)
pred = render(subject, k_inv, rt_inv_pred, sdd, height, width)
plot_drr(torch.concat([true, pred]))
plt.show()

In [ ]:
LR_ROT = 5e-2
LR_XYZ = 1e+1
N_ITRS = 500

reg = Registration(subject, rt_inv_pred, k_inv, sdd, height, width)
ncc = NormalizedCrossCorrelation2d()

optim = torch.optim.Adam(
    [
        {"params": [reg._rot], "lr": LR_ROT},
        {"params": [reg._xyz], "lr": LR_XYZ},
    ],
    maximize=True,
)

losses = []
for idx in tqdm(range(N_ITRS)):
    optim.zero_grad()
    pred = reg()
    loss = ncc(true, pred)
    loss.backward()
    optim.step()

    losses.append(loss.item())
    if loss > 0.999:
        break

plt.plot(losses, label="nanodrr")
plt.xlabel("Iteration")
plt.ylabel("NCC")
plt.legend()
plt.show()